# Gemini を使用した LLM のプロンプト エンジニアリング

**学習目標**

1. Google Gen AI SDK を使用して Gemini を呼び出す方法を学ぶ
1. Gemini のパラメータを設定する方法を学ぶ
1. テキスト生成のためのプロンプト エンジニアリングを学ぶ
1. チャット アプリケーションのためのプロンプト エンジニアリングを学ぶ


Google Gen AI SDK を使用すると、Google の大規模言語モデル（LLM）である Gemini のインスタンスをテスト、カスタマイズ、デプロイして、アプリケーションで Gemini の機能を活用できます。Gemini モデル ファミリーは、テキスト補完、マルチターン チャット、テキスト埋め込みの生成をサポートしています。

このノートブックでは、SDK を使用して事前トレーニング済みの Gemini モデルにアクセスし、テキスト分類、要約、抽出、チャットなどのユースケースに対応する例を紹介します。

## セットアップ

In [ ]:
from google import genai
from google.genai.types import Content, Part
from IPython.display import Markdown

## テキスト生成

下のセルでは、Gen AI SDK を使用して Gemini からストリーム応答を生成するヘルパー関数 `generate`（`generate_content_stream` を使用）を実装しています。<br>
ストリーミング出力が不要な場合は、代わりに `generate_content` を使用します。

In [ ]:
client = genai.Client(vertexai=True, location="global")

In [ ]:
def generate(
    prompt,
    model_name="gemini-3.5-flash",
):
    responses = client.models.generate_content_stream(
        model=model_name, contents=prompt
    )
    return responses

In [ ]:
responses = generate(
    "What are five important things to understand about large language models?"
)
for response in responses:
    print(response.text, end="")

### テキスト分類

ラッパー関数のテストが完了したので、分類のためのプロンプトを見ていきましょう。テキスト分類は機械学習における一般的なユースケースであり、スパム検出、感情分析、トピック分類などのタスクで頻繁に使用されます。

テキスト分類のユースケースでは、**zero-shot** プロンプトと **few-shot** プロンプトの両方がよく使用されます。zero-shot プロンプトは入力内にラベル付きの例を提供しない手法であり、few-shot プロンプトは入力内にいくつかの例を提供する手法です。

まず zero-shot 分類から始めましょう。

In [ ]:
prompt = """
Classify the following: 
text: "Zero-shot prompting is really easy with Google's Gen AI SDK."
label: technology, politics, sports 
"""

responses = generate(prompt)
for response in responses:
    print(response.text, end="")

次に、分類のための few-shot プロンプトの例を示します。few-shot プロンプトはモデルの精度を高めるだけでなく、出力形式をある程度制御することもできます。

In [ ]:
prompt = """
What is the topic for a given text? 
- cats 
- dogs 

Text: They always sits on my lap and purr!
The answer is: cats

Text: They love to play fetch
The answer is: dogs

Text: I throw the frisbee in the water and they swim for hours! 
The answer is: dogs 

Text: They always knock things off my counter!
The answer is:
"""

responses = generate(prompt)
for response in responses:
    print(response.text, end="")

### テキストの要約

LLM はテキスト要約のユースケースにも使用できます。テキスト要約では、より長いテキスト ドキュメントから簡潔で流暢な要約が生成されます。下のセルのプロンプトは、与えられたテキストを要約するよう Gemini にシンプルに指示しています。

In [ ]:
prompt = """
Provide a very short summary for the following:

A transformer is a deep learning model. It is distinguished by its adoption of self-attention, 
differentially weighting the significance of each part of the input (which includes the 
recursive output) data. Like recurrent neural networks (RNNs), transformers are designed to 
process sequential input data, such as natural language, with applications towards tasks such 
as translation and text summarization. However, unlike RNNs, transformers process the 
entire input all at once. The attention mechanism provides context for any position in the 
input sequence. For example, if the input data is a natural language sentence, the transformer 
does not have to process one word at a time. This allows for more parallelization than RNNs 
and therefore reduces training times.

Summary:

"""
responses = generate(prompt)
for response in responses:
    print(response.text, end="")

箇条書き形式など特定の形式で要約が必要な場合は、そのように Gemini に指示できます。

In [ ]:
prompt = """
Provide four bullet points summarizing the following:

A transformer is a deep learning model. It is distinguished by its adoption of self-attention, 
differentially weighting the significance of each part of the input (which includes the 
recursive output) data. Like recurrent neural networks (RNNs), transformers are designed to 
process sequential input data, such as natural language, with applications towards tasks such 
as translation and text summarization. However, unlike RNNs, transformers process the 
entire input all at once. The attention mechanism provides context for any position in the 
input sequence. For example, if the input data is a natural language sentence, the transformer 
does not have to process one word at a time. This allows for more parallelization than RNNs 
and therefore reduces training times.

Summary:

"""
responses = generate(prompt)
for response in responses:
    print(response.text, end="")

対話の要約（対話要約）もテキスト要約の範疇に含まれます。

In [ ]:
prompt = """
Generate a one-liner summary of the following chat and at the end, summarize to-do's for the service rep: 

Customer: Hi! I'm reaching out to customer service because I am having issues.

Service Rep: What seems to be the problem? 

Customer: I am trying to use the Gemini API but I keep getting an error. 

Service Rep: Can you share the error with me? 

Customer: Sure. The error says: "ResourceExhausted: 429 Quota exceeded for 
      aiplatform.googleapis.com/online_prediction_requests_per_base_model 
      with base model: text-bison"
      
Service Rep: It looks like you have exceeded the quota for usage. Please refer to 
             https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/quotas for information about quotas
             and limits. 
             
Customer: Can you increase my quota?

Service Rep: I cannot, but let me follow up with somebody who will be able to help.

Summary:
"""

responses = generate(prompt)
for response in responses:
    print(response.text, end="")

### テキストの抽出
Gemini は、テキストの抽出と構造化に使用できます。テキスト抽出はさまざまな目的に役立ちます。一般的な目的の 1 つは、ドキュメントを機械可読な形式に変換することです。これは、ドキュメントをデータベースに保存したり、ソフトウェアでドキュメントを処理したりする際に便利です。もう 1 つの一般的な目的は、ドキュメントから情報を抽出することです。これは、ドキュメント内の特定の情報を見つけたり、ドキュメントの内容を要約したりするのに役立ちます。

まず zero-shot 抽出から始めましょう。

In [ ]:
prompt = """
Extract the ingredients from the following recipe. 
Return the ingredients in JSON format with keys: ingredient, quantity, type.

Ingredients:
* 1 tablespoon olive oil
* 1 onion, chopped
* 2 carrots, chopped
* 2 celery stalks, chopped
* 1 teaspoon ground cumin
* 1/2 teaspoon ground coriander
* 1/4 teaspoon turmeric powder
* 1/4 teaspoon cayenne pepper (optional)
* Salt and pepper to taste
* 1 (15 ounce) can black beans, rinsed and drained
* 1 (15 ounce) can kidney beans, rinsed and drained
* 1 (14.5 ounce) can diced tomatoes, undrained
* 1 (10 ounce) can diced tomatoes with green chilies, undrained
* 4 cups vegetable broth
* 1 cup chopped fresh cilantro
"""
responses = generate(prompt)
for response in responses:
    print(response.text, end="")

分類と同様に、few-shot プロンプトを使用すると抽出される形式をより細かく制御できます。

In [ ]:
prompt = """
Extract the technical specifications from the text below in JSON format.

Text: Google Nest WiFi, network speed up to 1200Mpbs, 2.4GHz and 5GHz frequencies, WP3 protocol
JSON: {
  "product":"Google Nest WiFi",
  "speed":"1200Mpbs",
  "frequencies": ["2.4GHz", "5GHz"],
  "protocol":"WP3"
}

Text: Google Pixel 7, 5G network, 8GB RAM, Tensor G2 processor, 128GB of storage, Lemongrass
JSON:
"""

responses = generate(prompt)
for response in responses:
    print(response.text, end="")

## チャットのためのプロンプト エンジニアリング

チャット向けの Gen AI SDK は、マルチターン チャット用に最適化されています。マルチターン チャットとは、モデルがチャット会話の履歴を追跡し、その履歴を応答のコンテキストとして使用する機能です。

Gemini では、複数ターンにわたる自由形式の会話が可能です。`Chat` クラスは会話の状態を管理することでプロセスを簡素化するため、`generate_content` とは異なり、会話の履歴をリストとして手動で保持する必要がありません。

チャットを初期化してみましょう。

In [ ]:
chat = client.chats.create(model="gemini-3.5-flash")
chat

`Chat.send_message` メソッドは、`client.models.generate_content` と同じ `GenerateContentResponse` 型を返します。また、送信したメッセージと応答をチャット履歴に追加します。

In [ ]:
response = chat.send_message(
    "In one sentence, explain how a computer works to a young child."
)
response.text

チャット セッション内では履歴が保持されることを思い出してください。これにより、モデルはコンテキストのために特定のチャット セッション内の事項を記憶できます。この履歴は、チャット セッション オブジェクトの `get_history()` メソッドで取得できます。履歴は単に以前の入力と出力のペアのリストであることに注目してください。

In [ ]:
chat.get_history()

In [ ]:
response = chat.send_message("What question did I ask you last?")
response.text

### チャット履歴の追加

user ロールと model ロールのメッセージを交互に追加することで、チャットにチャット履歴を追加できます。システム メッセージは、最初のメッセージの最初のパートで設定できます。

In [ ]:
chat2 = client.chats.create(
    model="gemini-3.5-flash",
    history=[
        Content(
            role="user",
            parts=[
                Part.from_text(
                    text="""My name is Ned. You are my personal assistant. My favorite movies are Lord of the Rings and Hobbit.Who do you work for?"""
                )
            ],
        ),
        Content(role="model", parts=[Part.from_text(text="I work for Ned.")]),
        Content(role="user", parts=[Part.from_text(text="What do I like?")]),
        Content(
            role="model",
            parts=[Part.from_text(text="Ned likes watching movies.")],
        ),
    ],
)

response = chat2.send_message("Are my favorite movies based on a book series?")
Markdown(response.text)

In [ ]:
response = chat2.send_message("When were these books published?")
Markdown(response.text)

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.